# StepDAG - Build DAGs Programmatically

This notebook demonstrates the high-level StepDAG API for building workflows using Step objects.

## Example 1: Simple ETL Pipeline

In [1]:
from ipydagflow import Step, StepDAG

# Create steps
extract = Step(id="extract", label="Extract Data", step_type="datasource")
transform = Step(id="transform", label="Transform", step_type="box")
load = Step(id="load", label="Load to DB", step_type="datasource")

# Connect steps
extract.add_child(transform)
transform.add_child(load)

# Create DAG and render
dag = StepDAG()
dag.add_steps(extract, transform, load)
dag.render()

## Example 2: Multiple Data Sources

In [2]:
# Create steps
db = Step(id="db", label="PostgreSQL", step_type="datasource")
api = Step(id="api", label="REST API", step_type="datasource")
merge = Step(id="merge", label="Merge Data", step_type="box")
validate = Step(id="validate", label="Validate", step_type="box")
warehouse = Step(id="warehouse", label="Data Warehouse", step_type="datasource")

# Connect: both db and api feed into merge
db.add_child(merge)
api.add_child(merge)
merge.add_child(validate)
validate.add_child(warehouse)

# Render
dag = StepDAG()
dag.add_steps(db, api, merge, validate, warehouse)
dag.render()

## Example 3: Chaining with add_child()

You can chain operations for more concise code.

In [3]:
# Create and connect in one go
source = Step(id="source", label="Data Source", step_type="datasource")
clean = Step(id="clean", label="Clean Data", step_type="box")
enrich = Step(id="enrich", label="Enrich", step_type="box")
aggregate = Step(id="aggregate", label="Aggregate", step_type="box")
sink = Step(id="sink", label="Output", step_type="datasource")

# Chain: source -> clean -> enrich -> aggregate -> sink
source.add_child(clean).add_child(enrich).add_child(aggregate).add_child(sink)

dag = StepDAG()
dag.add_steps(source, clean, enrich, aggregate, sink)
dag.render()

## Example 4: Multiple Children

Use `add_children()` to add multiple child steps at once.

In [4]:
# One source feeding multiple processes
main_source = Step(id="main_source", label="Main DB", step_type="datasource")
process_a = Step(id="process_a", label="Process A", step_type="box")
process_b = Step(id="process_b", label="Process B", step_type="box")
process_c = Step(id="process_c", label="Process C", step_type="box")

# Add multiple children at once
main_source.add_children(process_a, process_b, process_c)

dag = StepDAG()
dag.add_steps(main_source, process_a, process_b, process_c)
dag.render()

## Example 5: Edge Labels

Add labels to edges to describe the data flowing between steps.

In [5]:
# ETL pipeline with labeled edges
extract = Step(id="extract", label="Extract", step_type="datasource")
transform = Step(id="transform", label="Transform", step_type="box")
load = Step(id="load", label="Load", step_type="datasource")

# Add edge labels to describe data flow
extract.add_child(transform, edge_label="raw_data")
transform.add_child(load, edge_label="cleaned_data")

dag = StepDAG()
dag.add_steps(extract, transform, load)
dag.render()

In [6]:
# Multiple branches with labeled edges
source = Step(id="source", label="Data Source", step_type="datasource")
branch_a = Step(id="branch_a", label="Branch A", step_type="box")
branch_b = Step(id="branch_b", label="Branch B", step_type="box")
branch_c = Step(id="branch_c", label="Branch C", step_type="box")
sink = Step(id="sink", label="Merge", step_type="box")

# Label each branch edge
source.add_children(branch_a, branch_b, branch_c, edge_labels=["users", "orders", "products"])
branch_a.add_child(sink, edge_label="user_metrics")
branch_b.add_child(sink, edge_label="order_stats")
branch_c.add_child(sink, edge_label="inventory")

dag = StepDAG()
dag.add_steps(source, branch_a, branch_b, branch_c, sink)
dag.render()

## Example 6: Adding Custom Data to Steps

In [7]:
# Steps with additional metadata
fetch = Step(
    id="fetch",
    label="Fetch Orders",
    step_type="datasource",
    data={
        "connection": "postgres://localhost/orders",
        "table": "orders",
        "frequency": "daily"
    }
)

process = Step(
    id="process",
    label="Calculate Metrics",
    step_type="box",
    data={
        "metrics": ["total_revenue", "avg_order_value"],
        "window": "7d"
    }
)

store = Step(
    id="store",
    label="Store Results",
    step_type="datasource",
    data={
        "destination": "s3://metrics-bucket",
        "format": "parquet"
    }
)

fetch.add_child(process).add_child(store)

dag = StepDAG()
dag.add_steps(fetch, process, store)

# Click on nodes to see the custom data!
dag.render()

## Example 7: Custom Styling

In [8]:
# Custom color scheme
custom_styles = {
    "datasource": {
        "background": "rgba(255, 99, 132, 0.8)",  # Red
        "color": "white",
        "borderColor": "#ff6384",
        "borderWidth": 3,
    },
    "box": {
        "background": "rgba(54, 162, 235, 0.8)",  # Blue
        "color": "white",
        "borderColor": "#36a2eb",
        "borderWidth": 2,
    },
}

# Create pipeline
s1 = Step(id="s1", label="Source", step_type="datasource")
s2 = Step(id="s2", label="Process", step_type="box")
s3 = Step(id="s3", label="Sink", step_type="datasource")
s1.add_child(s2).add_child(s3)

# Apply custom styles
dag = StepDAG(styles=custom_styles)
dag.add_steps(s1, s2, s3)
dag.render()

## Example 8: Complex Workflow

In [9]:
# Build a more complex DAG
raw_data = Step(id="raw", label="Raw Data", step_type="datasource")
clean_data = Step(id="clean", label="Clean", step_type="box")
feature_a = Step(id="feat_a", label="Feature A", step_type="box")
feature_b = Step(id="feat_b", label="Feature B", step_type="box")
feature_c = Step(id="feat_c", label="Feature C", step_type="box")
combine = Step(id="combine", label="Combine Features", step_type="box")
train = Step(id="train", label="Train Model", step_type="box")
evaluate = Step(id="eval", label="Evaluate", step_type="box")
deploy = Step(id="deploy", label="Deploy", step_type="datasource")

# Build relationships
raw_data.add_child(clean_data)
clean_data.add_children(feature_a, feature_b, feature_c)
feature_a.add_child(combine)
feature_b.add_child(combine)
feature_c.add_child(combine)
combine.add_child(train)
train.add_child(evaluate)
evaluate.add_child(deploy)

# Create and render
dag = StepDAG()
dag.add_steps(
    raw_data, clean_data, feature_a, feature_b, feature_c,
    combine, train, evaluate, deploy
)
dag.render()

## Example 9: Inspecting the DAG

In [10]:
# Create a simple DAG
s1 = Step(id="step1", label="Step 1", step_type="datasource")
s2 = Step(id="step2", label="Step 2", step_type="box")
s3 = Step(id="step3", label="Step 3", step_type="datasource")
s1.add_child(s2).add_child(s3)

dag = StepDAG()
dag.add_steps(s1, s2, s3)

# Inspect the DAG
print("DAG Info:", dag)
print("\nRoot steps:", dag.get_root_steps())
print("Leaf steps:", dag.get_leaf_steps())
print("\nAll steps:")
for step in dag.get_all_steps():
    print(f"  {step}")
    print(f"    Parents: {step.parents}")
    print(f"    Children: {step.children}")

DAG Info: StepDAG(steps=3, subflows=0, roots=1, leaves=1)

Root steps: [Step(id='step1', label='Step 1', type='datasource')]
Leaf steps: [Step(id='step3', label='Step 3', type='datasource')]

All steps:
  Step(id='step1', label='Step 1', type='datasource')
    Parents: []
    Children: [Step(id='step2', label='Step 2', type='box')]
  Step(id='step2', label='Step 2', type='box')
    Parents: [Step(id='step1', label='Step 1', type='datasource')]
    Children: [Step(id='step3', label='Step 3', type='datasource')]
  Step(id='step3', label='Step 3', type='datasource')
    Parents: [Step(id='step2', label='Step 2', type='box')]
    Children: []


## Example 10: Validation and Error Handling

In [11]:
# Try to create a cycle (will raise an error)
try:
    a = Step(id="a", label="A", step_type="box")
    b = Step(id="b", label="B", step_type="box")
    c = Step(id="c", label="C", step_type="box")

    # Create a cycle: a -> b -> c -> a
    a.add_child(b)
    b.add_child(c)
    c.add_child(a)  # This creates a cycle!

    dag = StepDAG()
    dag.add_steps(a, b, c)
    dag.render()  # This will raise ValueError
except ValueError as e:
    print(f"Error caught: {e}")

## API Summary

### Step Class
```python
step = Step(
    id="unique_id",           # Required: unique identifier
    label="Display Name",     # Required: display label
    step_type="box",          # Optional: datasource, box, or custom
    data={...}                # Optional: additional metadata
)

# Methods
step.add_child(other_step, edge_label="label")   # Add child with optional edge label
step.add_children(s1, s2, s3, edge_labels=["a", "b", "c"])  # Add multiple with labels
step.add_parent(parent_step)                      # Add parent
step.get_edge_label(child_id)                     # Get edge label for a child
step.get_all_descendants()                        # Get all descendants
step.get_all_ancestors()                          # Get all ancestors
```

### StepDAG Class
```python
dag = StepDAG(styles={...})        # Optional custom styles

# Methods
dag.add_step(step)                 # Add single step
dag.add_steps(s1, s2, s3)          # Add multiple steps
dag.get_step("step_id")            # Get step by ID
dag.get_all_steps()                # Get all steps
dag.get_root_steps()               # Get root steps
dag.get_leaf_steps()               # Get leaf steps
dag.validate()                     # Validate DAG structure
dag.render()                       # Render as DynamicDAG widget
```